In [27]:
# LAB 3: Prompt Engineering Patterns & Conversation Memory
# Google Colab + LangChain + Hugging Face
# STEP 1: Install required libraries
!pip install -q --upgrade langchain langchain-community transformers accelerate sentencepiece langchain-core

In [28]:
import requests
print(requests.__version__)

2.32.5


In [33]:
import torch
from langchain_community.llms import HuggingFacePipeline

# Initialize the HuggingFacePipeline directly from model_id
# Explicitly specify the task as 'text2text-generation' as this is a T5 model
llm = HuggingFacePipeline.from_model_id(
    model_id="google/flan-t5-base",
    task="text2text-generation",
    device=0 if torch.cuda.is_available() else -1, # Pass device directly to from_model_id
    pipeline_kwargs={
        "max_new_tokens": 256,
    }
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
# STEP 3: Import PromptTemplate and Memory components
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Try importing memory - handle different LangChain versions
try:
    from langchain.memory import ConversationBufferMemory
    from langchain.chains import ConversationChain
except ImportError:
    try:
        from langchain_community.memory import ConversationBufferMemory
        from langchain.chains import ConversationChain
    except ImportError:
        print("Note: Using alternative memory approach")
        ConversationBufferMemory = None
        ConversationChain = None

Note: Using alternative memory approach


In [18]:
# STEP 4: Demonstration WITHOUT Memory (Problem)
simple_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the question:\n{question}"
)

print("=== WITHOUT MEMORY ===")
print("Q: Who is Elon Musk?")
print("A:", llm.invoke(simple_prompt.format(
    question="Who is Elon Musk?"
)))

print("\nQ: What is his company?")
print("A:", llm.invoke(simple_prompt.format(
    question="What is his company?"
)))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== WITHOUT MEMORY ===
Q: Who is Elon Musk?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Answer the question:
Who is Elon Musk?

Q: What is his company?
A: Answer the question:
What is his company?


In [ ]:
conversation_history = []

def ask_with_memory(question):
    """Ask a question with conversation memory"""
    # Build context from conversation history
    history_context = ""
    if conversation_history:
        history_context = "\n\nPrevious conversation:\n"
        for i, (q, a) in enumerate(conversation_history, 1):
            history_context += f"Q{i}: {q}\nA{i}: {a}\n"

    # Create prompt with history
    prompt_text = f"""You are a helpful assistant. Use the conversation history to answer questions.{history_context}

Current question: {question}

Answer:"""

    # Get response
    response = llm.invoke(prompt_text)

    # Store in memory
    conversation_history.append((question, response))

    return response

# Clear previous conversation
conversation_history = []

print("=== WITH MEMORY ===")
print("Q: Who is Elon Musk?")
response1 = ask_with_memory("Who is Elon Musk?")
print("A:", response1)

print("\nQ: What company did he found?")
response2 = ask_with_memory("What company did he found?")
print("A:", response2)

print("\nQ: What field does that company work in?")
response3 = ask_with_memory("What field does that company work in?")
print("A:", response3)


=== WITH MEMORY ===
Q: Who is Steve Jobs?
A: Steve Jobs

Q: What is his wife name?
A: Sarah Michelle Gellar

Q: What company did he co-found?
A: Apple Inc.
